# Training a ResNet on Tiny ImageNet for 200 Epochs

## What This Notebook Does

This notebook trains a **pre-activation ResNet** classifier on the Tiny ImageNet dataset for **200 epochs** - significantly longer than previous notebooks (which used 25-50 epochs). The goal is to see how much accuracy we can achieve with extended training.

### Why Train for 200 Epochs?

| Epochs | Typical Result | Trade-off |
|--------|----------------|----------|
| 25 | ~50% accuracy | Quick experimentation |
| 50 | ~55-58% accuracy | Good balance |
| **200** | **~62-65% accuracy** | **Best accuracy, long training time** |

With strong data augmentation (TrivialAugmentWide), longer training continues to improve accuracy without severe overfitting.

### What You'll Learn

1. **Dataset preparation** for Tiny ImageNet
2. **Pre-activation ResNet** architecture
3. **TrivialAugmentWide** data augmentation
4. **Extended training** with OneCycleLR
5. **Model saving** for later use

### Prerequisites

- Basic Python knowledge
- Understanding of neural network basics (what training means)
- Previous notebooks in this series cover concepts in more detail

## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library.

**GPU note:** 200-epoch training is slow even on a GPU; budget ~hours on a T4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
!pip install -q fastcore fastai diffusers datasets torcheval accelerate wandb
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))
try:
    import miniai
    print(f'miniai loaded from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found.')

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`24a_imgnet_tiny-200_explained.ipynb`), unchanged.*

---

---
## Section 1: Environment Setup

First, we set up our computing environment by selecting the GPU and importing necessary libraries.

### 1.1 GPU Selection

In [ ]:
# Import the operating system module
import os

# Select which GPU to use by setting an environment variable
# '1' means use GPU #1 (the second GPU, since counting starts at 0)
# Change this to '0' if you only have one GPU
#
# IMPORTANT: This line MUST come BEFORE importing torch!
# Once torch is imported, it scans for GPUs and this setting is locked in.
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

**What is CUDA_VISIBLE_DEVICES?**

CUDA is NVIDIA's platform for GPU computing. When you have multiple GPUs, this environment variable tells PyTorch which ones to "see". Setting it to '1' makes only GPU #1 visible, which is useful when:

- Other users are using GPU #0
- You want to run multiple experiments on different GPUs
- A specific GPU has more memory

### 1.2 Importing Libraries

In [ ]:
# ============================================================
# STANDARD PYTHON LIBRARIES
# ============================================================

import shutil    # For file operations (extracting zip files)
import timm      # PyTorch Image Models - pretrained models library
import os        # Operating system interface
import torch     # PyTorch - the deep learning framework
import random    # Random number generation
import datasets  # HuggingFace datasets library
import math      # Mathematical functions

# ============================================================
# DATA & VISUALIZATION LIBRARIES
# ============================================================

# fastcore: Utility functions that extend Python's capabilities
import fastcore.all as fc

# numpy: Numerical computing (arrays, math operations)
import numpy as np

# matplotlib: Plotting and visualization
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# DEEP LEARNING LIBRARIES
# ============================================================

# k_diffusion: Diffusion model utilities (we use some helpers)
import k_diffusion as K

# torchvision.transforms: Image augmentation and preprocessing
import torchvision.transforms as T

# Functional transforms (same operations as T but as functions)
import torchvision.transforms.functional as TF

# Neural network functions (loss functions, activations, etc.)
import torch.nn.functional as F

# ============================================================
# SPECIFIC IMPORTS
# ============================================================

# DataLoader: Loads data in batches for efficient training
# default_collate: Combines individual samples into batches
from torch.utils.data import DataLoader, default_collate

# Path: Object-oriented filesystem paths (easier than string manipulation)
from pathlib import Path

# init: Weight initialization utilities
from torch.nn import init

# L: Enhanced list class from fastcore
from fastcore.foundation import L

# nn: Neural network building blocks (layers)
# tensor: Creates PyTorch tensors (like numpy arrays but GPU-compatible)
from torch import nn, tensor

# itemgetter: Efficiently extracts items from collections
from operator import itemgetter

# MulticlassAccuracy: Computes classification accuracy
from torcheval.metrics import MulticlassAccuracy

# partial: Creates functions with pre-filled arguments
from functools import partial

# lr_scheduler: Learning rate scheduling during training
from torch.optim import lr_scheduler

# optim: Optimization algorithms (SGD, Adam, etc.)
from torch import optim

# Fast image reading utilities
from torchvision.io import read_image, ImageReadMode

# glob: Find files matching patterns (like *.jpg)
from glob import glob

# ============================================================
# MINIAI CUSTOM MODULES
# These were built in earlier notebooks of this course
# ============================================================

from miniai.datasets import *     # Dataset utilities
from miniai.conv import *         # Convolution helpers
from miniai.learner import *      # Training loop
from miniai.activations import *  # Activation functions (GeneralRelu)
from miniai.init import *         # Weight initialization
from miniai.sgd import *          # Optimizers
from miniai.resnet import *       # ResNet building blocks
from miniai.augment import *      # Data augmentation (RandErase)
from miniai.accel import *        # Acceleration (mixed precision)
from miniai.training import *     # Training utilities

In [ ]:
# Progress bars for training visualization
from fastprogress import progress_bar

### 1.3 Configuration Settings

In [ ]:
# ============================================================
# DISPLAY SETTINGS
# ============================================================

# Configure how PyTorch prints tensors:
# - precision=5: Show 5 decimal places (e.g., 0.12345 not 0.1234567890)
# - linewidth=140: Allow wider lines before wrapping
# - sci_mode=False: Show 0.00001 instead of 1e-5
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)

# Set random seed for reproducibility
# This ensures we get the same "random" numbers each run
torch.manual_seed(1)

# Set figure resolution for matplotlib (70 DPI is good for notebooks)
mpl.rcParams['figure.dpi'] = 70

# ============================================================
# REPRODUCIBILITY
# ============================================================

# Set ALL random seeds (Python, NumPy, PyTorch) for full reproducibility
set_seed(42)

# ============================================================
# PERFORMANCE SETTINGS
# ============================================================

# Limit CPU workers to prevent memory issues
# More workers = faster data loading, but uses more RAM
if fc.defaults.cpus > 8: 
    fc.defaults.cpus = 8

---
## Section 2: Data Processing

We'll load the Tiny ImageNet dataset - a smaller version of ImageNet with:
- **200 classes** (categories of objects)
- **64x64 pixel** images
- **100,000 training** images (500 per class)
- **10,000 validation** images (50 per class)

### 2.1 Setting Up Data Paths

In [ ]:
# Create a Path object pointing to our data directory
# Path is from the pathlib module - it makes working with file paths easier
path_data = Path('data')

# Create the directory if it doesn't exist
# exist_ok=True prevents an error if the folder already exists
path_data.mkdir(exist_ok=True)

# Full path to Tiny ImageNet dataset
# The / operator with Path objects joins paths (like os.path.join)
path = path_data / 'tiny-imagenet-200'

### 2.2 Downloading the Dataset

In [ ]:
# URL where Tiny ImageNet is hosted (Stanford CS231n course)
url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'

# Only download if we don't already have the data
# path.exists() returns True if the folder exists
if not path.exists():
    # Download the zip file
    # fc.urlsave downloads from URL and saves to specified directory
    path_zip = fc.urlsave(url, path_data)
    
    # Extract the zip file
    # shutil.unpack_archive handles zip, tar, and other archive formats
    shutil.unpack_archive('data/tiny-imagenet-200.zip', 'data')

**Dataset Folder Structure:**

```
tiny-imagenet-200/
├── train/                      # Training images
│   ├── n01443537/              # Class folder (WordNet ID)
│   │   └── images/
│   │       └── *.JPEG          # 500 images per class
│   └── ... (200 class folders)
├── val/                        # Validation images
│   ├── images/                 # All 10,000 images in one folder
│   └── val_annotations.txt     # Maps filename -> class
├── wnids.txt                   # List of 200 class IDs
└── words.txt                   # Class ID -> human name
```

### 2.3 Setting Batch Size

In [ ]:
# Batch size: how many images to process at once
# Larger = faster training, but uses more GPU memory
# 512 is a good balance for 64x64 images on a modern GPU
bs = 512

### 2.4 Creating the Training Dataset Class

In [ ]:
class TinyDS:
    """
    Custom PyTorch Dataset for Tiny ImageNet training data.
    
    A PyTorch Dataset must implement:
    - __len__(): Returns the total number of samples
    - __getitem__(i): Returns the i-th sample as (input, target)
    
    For training data, the class label is encoded in the folder structure:
    path/n01443537/images/n01443537_0.JPEG
         ^^^^^^^^^
         This folder name IS the class label
    """
    
    def __init__(self, path):
        """
        Initialize the dataset by finding all image files.
        
        Args:
            path: Path to the data folder (e.g., 'data/tiny-imagenet-200/train')
        """
        # Store the path as a Path object
        self.path = Path(path)
        
        # Find all JPEG files in all subdirectories
        # Glob pattern explanation:
        #   **    = any number of subdirectories
        #   *     = any filename
        #   .JPEG = must end with .JPEG
        #   recursive=True = allow ** to match multiple levels
        self.files = glob(str(path / '**/*.JPEG'), recursive=True)
    
    def __len__(self): 
        """
        Return the total number of images.
        Called when you do len(dataset).
        """
        return len(self.files)
    
    def __getitem__(self, i): 
        """
        Return a single sample by index.
        Called when you do dataset[i].
        
        Args:
            i: Index of the sample (0 to len-1)
        
        Returns:
            Tuple of (file_path, class_id)
        """
        # Get the file path
        file_path = self.files[i]
        
        # Extract class from folder structure:
        # .../train/n01443537/images/n01443537_0.JPEG
        #           ^^^^^^^^^
        # Path(file_path).parent = 'images' folder
        # .parent.parent = class folder (n01443537)
        # .name = just the folder name, not full path
        class_id = Path(file_path).parent.parent.name
        
        return file_path, class_id

In [ ]:
# Create the training dataset
tds = TinyDS(path / 'train')

### 2.5 Creating the Validation Dataset

In [ ]:
# Path to validation annotations file
path_anno = path / 'val' / 'val_annotations.txt'

# Parse the annotations file into a dictionary
# File format: filename<TAB>class_id<TAB>bbox_info...
# We only need the first two columns
#
# Step by step:
# 1. path_anno.read_text() - read entire file as string
# 2. .splitlines() - split into list of lines
# 3. o.split('\t') - split each line by tab
# 4. [:2] - take first two items (filename, class_id)
# 5. dict(...) - convert to dictionary
anno = dict(o.split('\t')[:2] for o in path_anno.read_text().splitlines())

# Result: {'val_0.JPEG': 'n03444034', 'val_1.JPEG': 'n04067472', ...}

In [ ]:
class TinyValDS(TinyDS):
    """
    Dataset for Tiny ImageNet validation data.
    
    Inherits from TinyDS but overrides __getitem__ because
    validation images are organized differently:
    - All images in one folder (not organized by class)
    - Class labels are in val_annotations.txt file
    """
    
    def __getitem__(self, i): 
        """
        Return a validation sample.
        Looks up class in the annotations dictionary instead of folder structure.
        """
        file_path = self.files[i]
        
        # Get just the filename (e.g., 'val_123.JPEG')
        filename = os.path.basename(file_path)
        
        # Look up class in our annotations dictionary
        class_id = anno[filename]
        
        return file_path, class_id

In [ ]:
# Create validation dataset
vds = TinyValDS(path / 'val')

### 2.6 Transform Wrapper

In [ ]:
class TfmDS:
    """
    A wrapper that applies transforms to a dataset.
    
    This is the "decorator pattern" - we wrap an existing dataset
    and add functionality (transforms) without modifying the original.
    
    Args:
        ds: The base dataset to wrap
        tfmx: Transform function for x (images)
        tfmy: Transform function for y (labels)
    
    fc.noop is a "no operation" function that returns its input unchanged.
    It's used as default when no transform is needed.
    """
    
    def __init__(self, ds, tfmx=fc.noop, tfmy=fc.noop): 
        # Store dataset and transform functions
        self.ds = ds
        self.tfmx = tfmx
        self.tfmy = tfmy
    
    def __len__(self): 
        # Same length as wrapped dataset
        return len(self.ds)
    
    def __getitem__(self, i):
        # Get raw sample from base dataset
        x, y = self.ds[i]
        # Apply transforms and return
        return self.tfmx(x), self.tfmy(y)

### 2.7 Label Encoding

In [ ]:
# Load the list of 200 class IDs from wnids.txt
# Each line contains one WordNet ID (e.g., 'n01443537')
id2str = (path / 'wnids.txt').read_text().splitlines()

# Create reverse mapping: WordNet ID string -> integer index
# enumerate gives (0, 'n01443537'), (1, 'n01629819'), ...
# We flip to {'n01443537': 0, 'n01629819': 1, ...}
str2id = {v: k for k, v in enumerate(id2str)}

### 2.8 Normalization Statistics

In [ ]:
# Pre-computed mean and standard deviation for Tiny ImageNet
# These were calculated by averaging across the entire training set
# One value per color channel (Red, Green, Blue)

xmean = tensor([0.47565, 0.40303, 0.31555])  # Mean pixel values for R, G, B
xstd = tensor([0.28858, 0.24402, 0.26615])   # Standard deviations for R, G, B

# Note: Red has highest mean (0.476), Blue lowest (0.316)
# This tells us the dataset has a slight warm/reddish tint on average

**Why Normalize Images?**

Neural networks train better when inputs have:
- **Mean close to 0** (centered)
- **Standard deviation close to 1** (scaled)

The normalization formula is:
$$x_{normalized} = \frac{x - \mu}{\sigma}$$

Where $\mu$ is the mean and $\sigma$ is the standard deviation.

### 2.9 Transform Functions

In [ ]:
def tfmy(y): 
    """
    Transform for labels: Convert string class ID to integer tensor.
    
    Args:
        y: WordNet ID string (e.g., 'n01443537')
    
    Returns:
        Integer tensor (class index 0-199)
    
    Example:
        tfmy('n01443537') -> tensor(0)
        tfmy('n01629819') -> tensor(1)
    """
    return tensor(str2id[y])

In [ ]:
def denorm(x): 
    """
    Reverse normalization for displaying images.
    
    Takes a normalized tensor and converts back to [0, 1] range
    suitable for visualization.
    
    Formula: x_original = x_normalized * std + mean
    
    Args:
        x: Normalized image tensor
    
    Returns:
        Tensor with pixel values in [0, 1]
    """
    # Reverse the normalization
    # [:,None,None] adds dimensions for broadcasting: (3,) -> (3,1,1)
    result = x * xstd[:, None, None] + xmean[:, None, None]
    # Clip to valid range (some values might be slightly outside [0,1])
    return result.clip(0, 1)

### 2.10 Human-Readable Class Names

In [ ]:
# Load WordNet synset definitions from words.txt
# Format: n01443537<TAB>goldfish, Carassius auratus
all_synsets = [o.split('\t') for o in (path / 'words.txt').read_text().splitlines()]

# Create dictionary: WordNet ID -> human-readable name
# We take only the first name (before comma) and only our 200 classes
synsets = {
    k: v.split(',', maxsplit=1)[0] 
    for k, v in all_synsets 
    if k in id2str
}

# Result: {'n01443537': 'goldfish', 'n01629819': 'European fire salamander', ...}

### 2.11 Batch Transform Helper

In [ ]:
def tfm_batch(b, tfm_x=fc.noop, tfm_y=fc.noop): 
    """
    Apply transforms to a batch of data.
    
    Args:
        b: Batch tuple (inputs, targets)
        tfm_x: Transform for inputs
        tfm_y: Transform for targets
    
    Returns:
        Transformed batch (transformed_inputs, transformed_targets)
    """
    return tfm_x(b[0]), tfm_y(b[1])

---
## Section 3: Model Architecture Setup

Now we set up the components for our neural network.

### 3.1 Activation Function

In [ ]:
# GeneralRelu: A modified ReLU activation function
#
# Standard ReLU: f(x) = max(0, x)
#   Problem: If x < 0, gradient is 0 ("dead neurons")
#   Problem: Output is always positive (not centered)
#
# GeneralRelu: f(x) = max(leak*x, x) - sub
#   leak=0.1: Small gradient for negative inputs (prevents dead neurons)
#   sub=0.4: Shift output to center around 0
#
# partial() creates a new function with some arguments pre-set
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)

# Weight initialization function for leaky ReLU
# Kaiming initialization scaled for the leak factor
iw = partial(init_weights, leaky=0.1)

### 3.2 Network Configuration

In [ ]:
# Number of filters (channels) at each stage
# Each stage doubles the channels as we go deeper
# More channels = model can learn more features
nfs = (32, 64, 128, 256, 512, 1024)

### 3.3 Optimizer Configuration

In [ ]:
# AdamW optimizer with modified epsilon
# AdamW = Adam with decoupled weight decay (better regularization)
# eps=1e-5: Prevents division by zero (default 1e-8, but 1e-5 works better with mixed precision)
opt_func = partial(optim.AdamW, eps=1e-5)

### 3.4 Block Configuration

In [ ]:
# Number of ResBlocks at each stage
# More blocks = deeper network = can learn more complex patterns
# We use more blocks at early stages (higher resolution)
nbks = (3, 3, 2, 2, 1)  # 3+3+2+2+1 = 11 blocks total

---
## Section 4: Building the Pre-activation ResNet

This section defines the complete model architecture.

In [ ]:
def conv(ni, nf, ks=3, stride=1, act=nn.ReLU, norm=None, bias=True):
    """
    Create a pre-activation convolution block.
    
    Pre-activation order: Norm -> Activation -> Conv
    (Standard order is: Conv -> Norm -> Activation)
    
    Pre-activation improves gradient flow in deep networks.
    
    Args:
        ni: Number of input channels
        nf: Number of output channels
        ks: Kernel size (default 3x3)
        stride: Stride for spatial downsampling (default 1 = no downsampling)
        act: Activation function class (default ReLU)
        norm: Normalization layer class (default None)
        bias: Whether to include bias in conv (default True)
    
    Returns:
        nn.Sequential containing the layers
    """
    layers = []
    
    # Step 1: Normalization (on INPUT channels)
    if norm: 
        layers.append(norm(ni))
    
    # Step 2: Activation
    if act: 
        layers.append(act())
    
    # Step 3: Convolution
    # padding=ks//2 keeps spatial size same when stride=1
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    
    return nn.Sequential(*layers)


def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    """
    Create a double convolution block (the "body" of a ResBlock).
    
    Two convolutions:
    1. First conv: ni -> nf channels, stride=1
    2. Second conv: nf -> nf channels, with specified stride
    
    The stride (downsampling) happens in the second conv.
    """
    return nn.Sequential(
        conv(ni, nf, stride=1, act=act, norm=norm, ks=ks),       # Change channels
        conv(nf, nf, stride=stride, act=act, norm=norm, ks=ks)   # Maybe downsample
    )


class ResBlock(nn.Module):
    """
    Pre-activation Residual Block.
    
    The key idea of ResNets: output = F(x) + x
    The "+x" is the "skip connection" or "residual connection".
    
    Architecture:
    
            x (input)
               |
        +------+------+
        |             |
        v             v
    [convs]       [idconv]
    (main)        [pool]
        |         (skip)
        +------+------+
               |
               v
              (+)
               |
            output
    """
    
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        """
        Initialize the ResBlock.
        
        Args:
            ni: Input channels
            nf: Output channels
            stride: Downsampling factor (1=same size, 2=half size)
            ks: Kernel size
            act: Activation function
            norm: Normalization layer type
        """
        super().__init__()
        
        # Main path: two convolutions
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        
        # Skip path: 1x1 conv if channels change, else identity
        # fc.noop is a function that returns its input unchanged
        self.idconv = fc.noop if ni == nf else conv(ni, nf, ks=1, stride=1, act=None, norm=norm)
        
        # Downsampling for skip path: AvgPool if stride>1, else identity
        # We use AvgPool instead of strided conv (preserves more info)
        self.pool = fc.noop if stride == 1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x): 
        """
        Forward pass: output = main_path(x) + skip_path(x)
        """
        return self.convs(x) + self.idconv(self.pool(x))


def res_blocks(n_bk, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
    """
    Create a sequence of ResBlocks.
    
    Key design: Only the LAST block has stride>1 (for downsampling).
    Earlier blocks keep the same spatial size.
    
    Args:
        n_bk: Number of blocks
        ni: Input channels (first block)
        nf: Output channels (all blocks)
        stride: Stride for LAST block
        ks: Kernel size
        act: Activation function
        norm: Normalization layer
    
    Example with n_bk=3, ni=64, nf=128, stride=2:
        Block 0: 64->128, stride=1
        Block 1: 128->128, stride=1  
        Block 2: 128->128, stride=2  <- Only last block downsamples
    """
    return nn.Sequential(*[
        ResBlock(
            ni if i == 0 else nf,              # First block: ni, others: nf
            nf,                                 # All output nf channels
            stride=stride if i == n_bk-1 else 1,  # Only last block has stride
            ks=ks, 
            act=act, 
            norm=norm
        )
        for i in range(n_bk)
    ])


def get_dropmodel(act=act_gr, nfs=nfs, nbks=nbks, norm=nn.BatchNorm2d, drop=0.2):
    """
    Create the complete pre-activation ResNet model.
    
    Args:
        act: Activation function
        nfs: Tuple of channel counts per stage
        nbks: Tuple of block counts per stage
        norm: Normalization layer type
        drop: Dropout probability
    
    Returns:
        Complete model as nn.Sequential
    """
    layers = []
    
    # Initial convolution (no pre-activation - no prior features to normalize)
    # 3 RGB channels -> nfs[0] (32) channels
    # 5x5 kernel for larger initial receptive field
    layers.append(nn.Conv2d(3, nfs[0], 5, padding=2))
    
    # ResBlock stages
    # Each stage: nfs[i] -> nfs[i+1] channels with stride=2 (downsample)
    layers += [
        res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
        for i in range(len(nfs)-1)
    ]
    
    # Final activation and normalization
    # (needed because pre-activation puts these BEFORE conv, so last conv has no activation)
    layers.append(act_gr())
    layers.append(norm(nfs[-1]))
    
    # Global average pooling: (N, C, H, W) -> (N, C, 1, 1)
    layers.append(nn.AdaptiveAvgPool2d(1))
    
    # Flatten: (N, C, 1, 1) -> (N, C)
    layers.append(nn.Flatten())
    
    # Dropout for regularization
    layers.append(nn.Dropout(drop))
    
    # Classification head
    # Linear: 1024 -> 200 classes
    # bias=False because BatchNorm has its own bias
    layers.append(nn.Linear(nfs[-1], 200, bias=False))
    layers.append(nn.BatchNorm1d(200))
    
    # Combine and initialize weights
    return nn.Sequential(*layers).apply(iw)

**Network Architecture Summary:**

```
Layer              Output Shape      Channels
─────              ────────────      ────────
Input              (64, 64)          3 (RGB)
Conv 5x5           (64, 64)          32
Stage 1 (3 blocks) (32, 32)          64
Stage 2 (3 blocks) (16, 16)          128
Stage 3 (2 blocks) (8, 8)            256
Stage 4 (2 blocks) (4, 4)            512
Stage 5 (1 block)  (2, 2)            1024
AvgPool            (1, 1)            1024
Linear             200               200 classes
```

---
## Section 5: Training Callbacks and Metrics

In [ ]:
# Create metrics callback to track accuracy during training
# MulticlassAccuracy computes: correct_predictions / total_predictions
metrics = MetricsCB(accuracy=MulticlassAccuracy())

# Main callbacks for training
cbs = [
    DeviceCB(),              # Automatically move data to GPU
    metrics,                 # Track and display accuracy
    ProgressCB(plot=True),   # Show progress bar and loss plot
    MixedPrecision()         # Use FP16 for faster training
]

**What is Mixed Precision Training?**

| Precision | Bits | Memory | Speed | Use Case |
|-----------|------|--------|-------|----------|
| FP32 | 32 | Baseline | Baseline | Default |
| FP16 | 16 | 50% less | ~2x faster | Training |

MixedPrecision() automatically:
- Uses FP16 for most operations (forward pass, gradients)
- Keeps FP32 for sensitive operations (loss, weight updates)
- Handles gradient scaling to prevent underflow

---
## Section 6: Data Augmentation

Data augmentation creates variations of training images to prevent overfitting. With 200 epochs of training, strong augmentation is essential.

In [ ]:
# Advanced augmentation pipeline
aug_tfms = nn.Sequential(
    T.Pad(4),                   # Add 4 pixels of padding (64->72)
    T.RandomCrop(64),           # Random crop back to 64x64
    T.RandomHorizontalFlip(),   # 50% chance horizontal flip
    T.TrivialAugmentWide()      # Random augmentation from a set
)

# Separate normalization (applied after augmentation)
norm_tfm = T.Normalize(xmean, xstd)

# Random erase (cutout) augmentation
erase_tfm = RandErase()

**TrivialAugmentWide:**

Randomly applies ONE of these augmentations with random strength:
- Identity (no change)
- AutoContrast, Equalize
- Rotate, ShearX, ShearY
- TranslateX, TranslateY
- Brightness, Color, Contrast, Sharpness
- Posterize, Solarize

The simplicity (one augmentation per image) is what makes it effective - it provides diversity without destroying the image.

In [ ]:
# Import PIL for image handling
from PIL import Image

In [ ]:
def tfmx(x, aug=False):
    """
    Complete image transform pipeline.
    
    Steps:
    1. Load image from disk as PIL Image
    2. Apply augmentation (if training)
    3. Convert to tensor
    4. Normalize using dataset statistics
    5. Apply random erase (if training)
    
    Args:
        x: Path to image file
        aug: Whether to apply augmentation (True for training)
    
    Returns:
        Normalized tensor of shape (3, 64, 64)
    """
    # Step 1: Load as PIL Image
    # .convert('RGB') ensures 3 channels even for grayscale
    x = Image.open(x).convert('RGB')
    
    # Step 2: Apply augmentation (training only)
    if aug: 
        x = aug_tfms(x)
    
    # Step 3: Convert PIL Image to tensor
    # TF.to_tensor: (H,W,C) uint8 [0,255] -> (C,H,W) float [0,1]
    x = TF.to_tensor(x)
    
    # Step 4: Normalize
    x = norm_tfm(x)
    
    # Step 5: Random erase (training only)
    # RandErase expects batch dimension, so we add then remove it
    if aug: 
        x = erase_tfm(x[None])[0]  # x[None] adds dim, [0] removes it
    
    return x

### 6.1 Creating the Final Datasets

In [ ]:
# Create transformed datasets
# Training: WITH augmentation (aug=True)
# Validation: WITHOUT augmentation (aug=False, the default)
tfm_tds = TfmDS(tds, partial(tfmx, aug=True), tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)

In [ ]:
# Create DataLoaders
# get_dls creates train and validation loaders with proper shuffling
# bs=512: Process 512 images at a time
# num_workers=8: Use 8 CPU processes for parallel data loading
dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=8))

---
## Section 7: Model Creation Helper

In [ ]:
def get_model(): 
    """
    Create a model instance with our chosen dropout rate.
    
    We use drop=0.1 (10% dropout) which is relatively light.
    With 200 epochs and strong augmentation, we don't need heavy dropout.
    """
    return get_dropmodel(drop=0.1)

---
## Section 8: Visualizing Training Data

Let's see what our augmented training images look like.

In [ ]:
# Create a learner that runs just one batch (for visualization)
# SingleBatchCB stops training after one batch
learn = TrainLearner(
    get_model(),           # Create a model
    dls,                   # Our dataloaders
    F.cross_entropy,       # Loss function
    cbs=[SingleBatchCB(), DeviceCB()]  # Just run one batch
)

# Run one training step
learn.fit(1)

# Get the batch that was processed
xb, yb = learn.batch

# Display first 9 images (denormalized for viewing)
show_images(denorm(xb.cpu())[:9], imsize=2.5)

Notice the variety in the images:
- Different crops (from Pad + RandomCrop)
- Some flipped horizontally
- Various color/contrast changes (from TrivialAugmentWide)
- Some with erased rectangles (from RandErase)

This variety helps the model generalize better!

---
## Section 9: Training for 200 Epochs

Now we train for 200 epochs - this is the main training run that will take considerable time.

In [ ]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

# Number of epochs (full passes through the training data)
# 200 epochs is much longer than typical (25-50)
# With strong augmentation, longer training continues to improve accuracy
epochs = 200

# Learning rate - the step size for weight updates
# 0.1 is relatively high but works well with OneCycleLR
lr = 0.1

# Total training steps (for learning rate scheduler)
# Each epoch has len(dls.train) batches
tmax = epochs * len(dls.train)

# OneCycleLR scheduler: varies learning rate during training
# - Warmup: LR increases from lr/25 to max_lr
# - Annealing: LR decreases from max_lr to very small
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)

# Additional callback for learning rate scheduling
xtra = [BatchSchedCB(sched)]

# ============================================================
# CREATE THE LEARNER
# ============================================================

learn = Learner(
    get_model(),           # Fresh model instance
    dls,                   # DataLoaders
    F.cross_entropy,       # Loss function for classification
    lr=lr,                 # Learning rate
    cbs=cbs+xtra,          # All callbacks (metrics, progress, mixed precision, scheduler)
    opt_func=opt_func      # AdamW optimizer
)

**OneCycleLR Learning Rate Schedule:**

```
Learning Rate
    ^
    |       /\            Peak at ~30% of training
    |      /  \
    |     /    \
    |    /      \
    |   /        \______  Annealing to near-zero
    |  /
    +---------------------> Epoch
    0    40   100    200
```

**Why this works:**
1. **Warmup** (start low): Helps model find a good region before taking big steps
2. **High LR middle**: Allows escaping local minima, faster learning
3. **Annealing** (end low): Fine-tunes to find precise minimum

In [ ]:
# Train for 200 epochs!
# This will take several hours depending on your GPU
# Expected accuracy: ~62-65% on validation set
learn.fit(epochs)

**What to expect during training:**

| Epoch Range | Expected Behavior |
|-------------|-------------------|
| 0-20 | Loss drops rapidly, accuracy climbs |
| 20-60 | Peak learning rate, fast improvement |
| 60-150 | Steady improvement as LR decreases |
| 150-200 | Fine-tuning, small improvements |

The validation accuracy may fluctuate but should trend upward. With TrivialAugmentWide, the model continues learning even at 200 epochs without severe overfitting.

---
## Section 10: Saving the Trained Model

In [ ]:
# Save the trained model to disk
# This saves all the learned weights so we can:
# 1. Use the model later without retraining
# 2. Fine-tune on other tasks
# 3. Share with others
#
# The file will be ~40MB (depends on model size)
torch.save(learn.model, 'models/inettiny-trivaug-200')

**Loading the model later:**

```python
# To use this model in another notebook:
model = torch.load('models/inettiny-trivaug-200')
model.eval()  # Set to evaluation mode

# Make predictions
with torch.no_grad():
    predictions = model(images)
```

---
## Summary

### What We Accomplished

We trained a pre-activation ResNet on Tiny ImageNet for 200 epochs, achieving significantly higher accuracy than shorter training runs.

### Key Components

| Component | Configuration |
|-----------|---------------|
| **Dataset** | Tiny ImageNet (200 classes, 64x64) |
| **Architecture** | Pre-activation ResNet |
| **Channels** | (32, 64, 128, 256, 512, 1024) |
| **Blocks per stage** | (3, 3, 2, 2, 1) = 11 total |
| **Augmentation** | TrivialAugmentWide + RandErase |
| **Optimizer** | AdamW (eps=1e-5) |
| **Scheduler** | OneCycleLR (max_lr=0.1) |
| **Training** | 200 epochs, mixed precision |
| **Dropout** | 0.1 (10%) |

### Why 200 Epochs Works

1. **Strong augmentation** (TrivialAugmentWide) creates infinite variations
2. **OneCycleLR** prevents learning rate from being too high at end
3. **Dropout** and **BatchNorm** provide regularization
4. **Pre-activation** architecture enables stable training

### Results Comparison

| Epochs | Approximate Accuracy |
|--------|----------------------|
| 25 | ~50% |
| 50 | ~55-58% |
| 100 | ~60-62% |
| **200** | **~62-65%** |

### fin -